 Install dependencies

In [ ]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

print("✅ All dependencies installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 11.5 MB/s eta 0:00:00
✅ All dependencies installed successfully!


Imports and configuration

In [ ]:
import re
import ast
import torch
import gradio as gr
import black
import autopep8
from datetime import datetime
from typing import Tuple, Optional, List, Dict, Any
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
import io
import base64
import textwrap

# Configuration class
class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 64
    DEFAULT_TOP_P = 0.95

    # Detect device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"📊 Using device: {DEVICE}")

config = ChatbotConfig()

# Initialize conversation history
conversation_history = []

📊 Using device: cpu


Load model using HuggingFace Transformers pipeline

In [ ]:
print("🔄 Loading model and tokenizer...")

try:
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_NAME,
        torch_dtype=torch.float16 if config.DEVICE == "cuda" else torch.float32,
        device_map="auto" if config.DEVICE == "cuda" else None
    )

    # Create pipeline
    pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if config.DEVICE == "cuda" else -1
    )

    print("✅ Model loaded successfully!")

except Exception as e:
    print(f"❌ Error loading model: {str(e)}")
    raise

🔄 Loading model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Model loaded successfully!


Python Code Detection

In [ ]:
def detect_python_code(text: str) -> bool:
    """
    Detect if the text contains Python code using regex patterns.

    Args:
        text: Input text to check

    Returns:
        True if Python code is detected, False otherwise
    """
    # Common Python code patterns
    patterns = [
        r'^def\s+\w+\s*\(.*\)\s*:',  # Function definitions
        r'^class\s+\w+.*:',           # Class definitions
        r'^import\s+\w+',              # Import statements
        r'^from\s+\w+\s+import',       # From import statements
        r'^if\s+.*:',                   # If statements
        r'^for\s+.*:',                   # For loops
        r'^while\s+.*:',                 # While loops
        r'^try\s*:',                      # Try blocks
        r'^except\s*.*:',                 # Except blocks
        r'^with\s+.*:',                   # With statements
        r'^@\w+',                         # Decorators
        r'^print\s*\(',                    # Print statements
        r'^return\s+',                      # Return statements
        r'^\s*#.*',                          # Comments
        r'^\s*"""',                           # Docstrings
        r'^\s*def\s+__\w+__',                 # Special methods
        r'^\s*self\.',                         # Self references
        r'^\s*cls\.',                           # Class references
    ]

    # Split text into lines and check each line
    lines = text.strip().split('\n')

    for line in lines:
        line = line.rstrip()
        if line:  # Non-empty line
            for pattern in patterns:
                if re.match(pattern, line, re.MULTILINE):
                    return True

    # Check for Python keywords in context
    python_keywords = ['lambda', 'yield', 'assert', 'global', 'nonlocal', 'del']
    for keyword in python_keywords:
        if re.search(rf'\b{keyword}\b', text):
            return True

    return False

Syntax Checking

In [ ]:
def check_syntax(code: str) -> Tuple[bool, Optional[str]]:
    """
    Check Python code syntax using AST.

    Args:
        code: Python code to check

    Returns:
        Tuple of (is_valid, error_message)
    """
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"Syntax Error: {str(e)}"
    except Exception as e:
        return False, f"Error: {str(e)}"

Code Fixing System

In [ ]:
def fix_code_indentation(code: str) -> str:
    """
    Fix common indentation issues in Python code.

    Args:
        code: Python code to fix

    Returns:
        Fixed code with proper indentation
    """
    lines = code.split('\n')
    fixed_lines = []

    # Convert tabs to spaces (4 spaces per tab)
    for line in lines:
        # Replace tabs with 4 spaces
        line = line.replace('\t', '    ')
        # Remove trailing spaces
        line = line.rstrip()
        fixed_lines.append(line)

    return '\n'.join(fixed_lines)

def fix_code_brackets(code: str) -> str:
    """
    Auto-complete missing closing brackets.

    Args:
        code: Python code to fix

    Returns:
        Code with balanced brackets
    """
    brackets = {
        '(': ')',
        '[': ']',
        '{': '}'
    }

    stack = []
    fixed_code = []

    for char in code:
        fixed_code.append(char)
        if char in brackets.keys():
            stack.append(char)
        elif char in brackets.values():
            if stack and brackets[stack[-1]] == char:
                stack.pop()

    # Add missing closing brackets in reverse order
    while stack:
        fixed_code.append(brackets[stack.pop()])

    return ''.join(fixed_code)

def format_code(code: str) -> str:
    """
    Format code using Black formatter with fallback to autopep8.

    Args:
        code: Python code to format

    Returns:
        Formatted code
    """
    try:
        # Try Black formatter first
        formatted = black.format_str(code, mode=black.Mode())
        return formatted
    except:
        try:
            # Fallback to autopep8
            formatted = autopep8.fix_code(code)
            return formatted
        except Exception as e:
            print(f"⚠️ Formatting failed: {str(e)}")
            return code

def fix_python_code(code: str) -> Tuple[str, List[str]]:
    """
    Comprehensive code fixing function.

    Args:
        code: Python code to fix

    Returns:
        Tuple of (fixed_code, list_of_changes_made)
    """
    changes = []
    original_code = code

    # Fix indentation
    code = fix_code_indentation(code)
    if code != original_code:
        changes.append("✅ Fixed indentation issues")

    # Fix brackets
    code = fix_code_brackets(code)
    if code != original_code:
        changes.append("✅ Balanced brackets")

    # Format code
    formatted_code = format_code(code)
    if formatted_code != code:
        code = formatted_code
        changes.append("✅ Applied code formatting")

    # Check syntax after fixes
    is_valid, error = check_syntax(code)
    if not is_valid:
        changes.append(f"⚠️ Syntax check: {error}")

    return code, changes


Conversation History Management

In [ ]:
def add_to_history(role: str, content: str):
    """
    Add a message to conversation history.

    Args:
        role: 'user' or 'assistant'
        content: Message content
    """
    conversation_history.append({
        "role": role,
        "content": content,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

def clear_history():
    """Clear conversation history."""
    global conversation_history
    conversation_history = []
    return "✅ Conversation history cleared!"

def get_history_text() -> str:
    """
    Get conversation history as formatted text.

    Returns:
        Formatted conversation history
    """
    if not conversation_history:
        return "No conversation history yet."

    history_text = "THE SYNTAX SURGEON - CONVERSATION HISTORY\n"
    history_text += "=" * 50 + "\n\n"

    for msg in conversation_history:
        role = msg["role"].upper()
        content = msg["content"]
        timestamp = msg.get("timestamp", "Unknown time")

        history_text += f"[{timestamp}] {role}:\n"
        history_text += f"{content}\n"
        history_text += "-" * 30 + "\n"

    return history_text


Response Generation

In [ ]:
def generate_response(
    message: str,
    temperature: float,
    max_tokens: int,
    top_p: float
) -> Tuple[str, str]:
    """
    Generate response based on user message.

    Args:
        message: User input message
        temperature: Model temperature parameter
        max_tokens: Maximum tokens to generate
        top_p: Top-p sampling parameter

    Returns:
        Tuple of (assistant_response, debug_info)
    """
    try:
        # Add user message to history
        add_to_history("user", message)

        # Detect if message contains Python code
        contains_code = detect_python_code(message)
        debug_info = f"Code detected: {contains_code}\n"

        response_parts = []

        if contains_code:
            # Extract and fix code
            response_parts.append("🔍 I detected Python code in your message. Let me analyze and fix it:\n\n")

            # Fix the code
            fixed_code, changes = fix_python_code(message)

            if changes:
                response_parts.append("Changes made:\n")
                for change in changes:
                    response_parts.append(f"  {change}\n")
                response_parts.append("\n")

            response_parts.append("📝 Fixed code:\n")
            response_parts.append("```python\n")
            response_parts.append(fixed_code)
            response_parts.append("\n```\n\n")

            # Check final syntax
            is_valid, error = check_syntax(fixed_code)
            if is_valid:
                response_parts.append("✅ Syntax check passed!")
            else:
                response_parts.append(f"⚠️ {error}")

        # Generate explanation using the model
        prompt = f"User: {message}\nAssistant:"

        # Update pipeline parameters
        pipe.model.generation_config.temperature = temperature
        pipe.model.generation_config.top_p = top_p
        pipe.model.generation_config.max_new_tokens = max_tokens

        # Generate response
        result = pipe(prompt, max_new_tokens=80, do_sample=True)

        explanation = result[0]['generated_text'].replace(prompt, "").strip()

        if explanation:
            response_parts.append("\n\n💡 Explanation:\n")
            response_parts.append(explanation)

        final_response = "".join(response_parts)

        # Add assistant response to history
        add_to_history("assistant", final_response)

        return final_response, debug_info

    except Exception as e:
        error_msg = f"❌ Error generating response: {str(e)}"
        add_to_history("assistant", error_msg)
        return error_msg, f"Error: {str(e)}"


Export Features

In [ ]:
def download_txt():
    """Generate downloadable TXT file of conversation history."""
    try:
        history_text = get_history_text()

        # Create download link
        b64 = base64.b64encode(history_text.encode()).decode()
        href = f'<a href="data:text/plain;base64,{b64}" download="syntax_surgeon_history.txt">📥 Download TXT</a>'

        return href
    except Exception as e:
        return f"Error creating TXT: {str(e)}"

def download_pdf():
    """Generate downloadable PDF file of conversation history."""
    try:
        buffer = io.BytesIO()
        doc = SimpleDocTemplate(buffer, pagesize=letter)
        styles = getSampleStyleSheet()
        story = []

        # Custom styles
        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=24,
            textColor=colors.HexColor('#2E86AB'),
            alignment=1,  # Center alignment
            spaceAfter=30
        )

        user_style = ParagraphStyle(
            'UserStyle',
            parent=styles['Normal'],
            fontSize=12,
            textColor=colors.HexColor('#1E3D58'),
            backColor=colors.HexColor('#E8F0FE'),
            borderPadding=10,
            borderWidth=1,
            borderColor=colors.HexColor('#2E86AB'),
            borderRadius=5,
            spaceAfter=10
        )

        assistant_style = ParagraphStyle(
            'AssistantStyle',
            parent=styles['Normal'],
            fontSize=12,
            textColor=colors.HexColor('#0B3B3B'),
            backColor=colors.HexColor('#E8F8F5'),
            borderPadding=10,
            borderWidth=1,
            borderColor=colors.HexColor('#27AE60'),
            borderRadius=5,
            spaceAfter=10
        )

        # Add title
        title = Paragraph("The Syntax Surgeon - Conversation History", title_style)
        story.append(title)
        story.append(Spacer(1, 20))

        # Add timestamp
        timestamp = Paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Normal'])
        story.append(timestamp)
        story.append(Spacer(1, 30))

        # Add conversation messages
        for msg in conversation_history:
            role = msg["role"].upper()
            content = msg["content"]
            timestamp = msg.get("timestamp", "Unknown time")

            # Format message
            header = f"<b>[{timestamp}] {role}:</b><br/><br/>"

            # Wrap long lines
            wrapped_content = textwrap.fill(content, width=80)
            wrapped_content = wrapped_content.replace('\n', '<br/>')

            full_message = header + wrapped_content

            if role == "USER":
                p = Paragraph(full_message, user_style)
            else:
                p = Paragraph(full_message, assistant_style)

            story.append(p)
            story.append(Spacer(1, 15))

        # Build PDF
        doc.build(story)

        # Get PDF value
        pdf_value = buffer.getvalue()
        buffer.close()

        # Create download link
        b64 = base64.b64encode(pdf_value).decode()
        href = f'<a href="data:application/pdf;base64,{b64}" download="syntax_surgeon_history.pdf">📥 Download PDF</a>'

        return href
    except Exception as e:
        return f"Error creating PDF: {str(e)}"

def generate_share_link():
    """Generate a shareable text link of conversation."""
    try:
        history_text = get_history_text()

        # Create shareable link (simulated)
        import hashlib
        hash_id = hashlib.md5(history_text.encode()).hexdigest()[:8]

        share_content = f"""
        🌟 The Syntax Surgeon - Shared Conversation
        ID: {hash_id}
        Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

        {history_text}

        Share this link: https://syntax-surgeon.example/share/{hash_id}
        (Note: This is a simulation - in production, you'd upload to a server)
        """

        b64 = base64.b64encode(share_content.encode()).decode()
        href = f'<a href="data:text/plain;base64,{b64}" download="shared_conversation_{hash_id}.txt">📤 Download Shareable File</a>'

        return href + "<br/><br/>📋 Copy this share ID: " + hash_id
    except Exception as e:
        return f"Error generating share link: {str(e)}"


Gradio Interface

In [ ]:
# Create custom CSS for better styling
custom_css = """
    .gradio-container {
        max-width: 1200px !important;
        margin: auto !important;
    }
    .chatbot-container {
        height: 400px !important;
        overflow-y: auto !important;
    }
    .user-message {
        background-color: #e3f2fd !important;
        border-radius: 10px !important;
        padding: 10px !important;
        margin: 5px !important;
    }
    .assistant-message {
        background-color: #f1f8e9 !important;
        border-radius: 10px !important;
        padding: 10px !important;
        margin: 5px !important;
    }
"""

def create_interface():
    """Create the Gradio interface."""

    with gr.Blocks(css=custom_css, title="The Syntax Surgeon") as demo:
        gr.Markdown("""
        # 🏥 The Syntax Surgeon
        ### AI-Powered Code Fixer & Conversational Assistant
        """)

        with gr.Row():
            # Left column - Chat
            with gr.Column(scale=2):
                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=400,
                    elem_classes="chatbot-container"
                )

                with gr.Row():
                    msg_input = gr.Textbox(
                        label="Your Message",
                        placeholder="Ask a question or paste Python code...",
                        scale=4
                    )
                    send_btn = gr.Button("Send", variant="primary", scale=1)

                clear_btn = gr.Button("Clear Chat", variant="secondary")

            # Right column - Controls
            with gr.Column(scale=1):
                gr.Markdown("### ⚙️ Model Parameters")

                temperature = gr.Slider(
                    minimum=0.1,
                    maximum=1.0,
                    value=config.DEFAULT_TEMP,
                    step=0.05,
                    label="Temperature",
                    info="Higher = more creative, Lower = more focused"
                )

                max_tokens = gr.Slider(
                    minimum=50,
                    maximum=1024,
                    value=config.DEFAULT_MAX_TOKENS,
                    step=10,
                    label="Max Tokens",
                    info="Maximum length of response"
                )

                top_p = gr.Slider(
                    minimum=0.1,
                    maximum=1.0,
                    value=config.DEFAULT_TOP_P,
                    step=0.05,
                    label="Top-p",
                    info="Nucleus sampling parameter"
                )

                gr.Markdown("### 📁 Export Options")

                with gr.Row():
                    txt_btn = gr.Button("📄 TXT")
                    pdf_btn = gr.Button("📑 PDF")

                share_btn = gr.Button("🔗 Generate Share Link")

                output_html = gr.HTML(label="Output")
                debug_info = gr.Textbox(label="Debug Info", visible=False)

        # State for conversation
        state = gr.State([])

        # Event handlers
        def respond(message, chat_history, temp, max_tok, top_p_val):
            """Handle user message and generate response."""
            response, debug = generate_response(message, temp, max_tok, top_p_val)
            chat_history.append((message, response))
            return chat_history, chat_history, debug

        send_btn.click(
            respond,
            inputs=[msg_input, state, temperature, max_tokens, top_p],
            outputs=[chatbot, state, debug_info]
        ).then(
            lambda: "", None, msg_input  # Clear input after sending
        )

        msg_input.submit(
            respond,
            inputs=[msg_input, state, temperature, max_tokens, top_p],
            outputs=[chatbot, state, debug_info]
        ).then(
            lambda: "", None, msg_input
        )

        def clear_chat():
            """Clear the chat history."""
            clear_history()
            return [], []

        clear_btn.click(
            clear_chat,
            outputs=[chatbot, state]
        )

        txt_btn.click(
            download_txt,
            outputs=[output_html]
        )

        pdf_btn.click(
            download_pdf,
            outputs=[output_html]
        )

        share_btn.click(
            generate_share_link,
            outputs=[output_html]
        )

    return demo

Launch Application

In [ ]:
print("🎯 Initializing The Syntax Surgeon...")
print("=" * 50)
print(f"Model: {config.MODEL_NAME}")
print(f"Device: {config.DEVICE}")
print(f"Default Temperature: {config.DEFAULT_TEMP}")
print(f"Default Max Tokens: {config.DEFAULT_MAX_TOKENS}")
print("=" * 50)

# Create and launch the interface
demo = create_interface()

# Launch with public link
print("\n🚀 Launching Gradio interface...")
print("⏱️ This may take a few seconds...")
print("=" * 50)

demo.launch(
    share=True,
    debug=False,
    server_name="0.0.0.0"
)

print("\n✅ Application is running!")
print("📊 Check the output above for the public share link")

🎯 Initializing The Syntax Surgeon...
Model: ibm-granite/granite-3.3-2b-instruct
Device: cpu
Default Temperature: 0.7
Default Max Tokens: 64


/tmp/ipykernel_713/715707851.py:28: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="The Syntax Surgeon") as demo:
/tmp/ipykernel_713/715707851.py:37: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_713/715707851.py:37: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(



🚀 Launching Gradio interface...
⏱️ This may take a few seconds...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2aff471189a3ce0e1b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ Application is running!
📊 Check the output above for the public share link
